# Analýza migrace a demografické struktury cizinců v ČR (2004–2024)

### Cíl projektu
Tento notebook se zaměřuje na detailní analýzu dat Českého statistického úřadu (ČSÚ) o cizincích na území České republiky. Cílem je identifikovat dlouhodobé trendy, změny v národnostním složení a regionální distribuci cizinců.

**Postup analýzy jsme rozdělili do následujících kroků:**

1.  **Příprava dat:** Načtení, čištění a sjednocení ročních výkazů.
2.  **Časové řady:** Analýza celkového vývoje a trendů jednotlivých národností.
3.  **Regionální analýza:** Zkoumání distribuce v krajích a okresech.
4.  **Geoprostorová vizualizace:** Tvorba interaktivních map (původ, pohlaví, hustota).
5.  **Analýza vzdálenosti:** Zkoumání vlivu vzdálenosti od hranic na hustotu osídlení u sousedních států.

### Metodika a data
* **Zdroj dat:** Veřejná databáze ČSÚ (Cizinci podle státního občanství, věku a pohlaví).
* **Nástroje:** Python (Pandas pro manipulaci s daty, Matplotlib/Folium pro vizualizaci).
* **Přístup:** Data jsou čištěna, agregována podle roků/zemí a následně vizualizována pomocí statických grafů a interaktivních map.

### Příprava datového prostředí

Než začneme s analýzou, musíme sjednotit roztříštěné datové zdroje do jednoho funkčního celku.

**Postup zpracování dat:**

1.  Načetli jsme hromadně všechny CSV soubory ze složky `data/`.
2.  Spojili jsme je do jednoho velkého DataFrame (`df_all`).
3.  Ošetřili jsme chybějící hodnoty (NaN nahrazeny nulou) a převedli sloupce na správné číselné typy.
4.  Připravili jsme proměnnou `max_rok` pro automatickou detekci posledního dostupného období.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
import glob
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
import io
import itertools
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

In [ ]:
# NAČTENÍ DAT
soubory = glob.glob("data/*.csv") 
dataframes = []

for f in soubory:
    df = pd.read_csv(f)
    dataframes.append(df)

df_all = pd.concat(dataframes, ignore_index=True)
# NaN nahradíme nulou
df_all['hodnota'] = pd.to_numeric(df_all['hodnota'], errors='coerce').fillna(0)

max_rok = 2024


### Jak se vyvíjí celkový počet cizinců v čase?

Následující graf zobrazuje celkovou křivku počtu cizinců v ČR za posledních 20 let. 

**Postup přípravy grafu:**

1.  Vyfiltrovali jsme data pro celou Českou republiku (kód 97).
2.  Agregovali jsme celkové součty bez rozlišení pohlaví, věku či národnosti.
3.  Seřadili jsme data chronologicky podle roků.
4.  **Výsledek jsme vizualizovali:**
    * **Modrá křivka:** Celkový počet cizinců legálně pobývajících na území ČR v daném roce.

In [ ]:
# PŘÍPRAVA DAT PRO GRAF 1
# Data pro celou ČR v čase
df_cz_total = df_all[
    (df_all['vuzemi_cis'] == 97) &       # 97 = Kód pro celou Českou republiku
    (df_all['pohlavi_txt'].isna()) &     # isna() = nevyplněno (chceme součet za obě pohlaví)
    (df_all['vek_txt'].isna()) &         # isna() = nevyplněno (chceme součet za všechny věky)
    (df_all['stobcan_txt'].isna())       # isna() = nevyplněno (chceme součet za všechny cizince)
].copy().sort_values('rok')              # Seřadíme chronologicky podle roku


In [ ]:
# VYKRESLENÍ GRAFU 1
# Vývoj celkového počtu cizinců
fig, ax = plt.subplots(figsize=(10, 5))
df_cz_total.plot(x='rok', y='hodnota', kind='line', ax=ax, marker='o', linewidth=2.5, color='darkblue', legend=False)
ax.set_title('Vývoj celkového počtu cizinců v ČR')
ax.set_ylabel('Počet osob')
ax.set_xlabel('Rok')
# Nastavení titulků a popisů os
# StrMethodFormatter pro oddělovače tisíců
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
# Celá čísla pro roky
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.grid(True)
plt.tight_layout()
plt.show()


### Které národnosti jsou u nás nejpočetnější?

Většinu cizinců v České republice tvoří občané jen několika málo států. Abychom těmto klíčovým skupinám lépe porozuměli, izolujeme pět nejpočetnějších národností (k aktuálnímu roku) a podíváme se na jejich vývoj v čase odděleně. To nám umožní vidět dynamiku hlavních migračních proudů bez šumu menších skupin.

**Postup analýzy:**

1.  Identifikovali jsme 5 zemí s nejvyšším počtem občanů v aktuálním roce (2024).
2.  Vyfiltrovali jsme historická data pouze pro těchto 5 států.
3.  Vytvořili jsme kontingenční tabulku (pivot), kde sloupce tvoří jednotlivé země.
4.  **Výsledek jsme vizualizovali:**
    * **Jednotlivé linie:** Představují vývoj počtu občanů dané země v čase.
    * **Legenda:** Seřazena podle aktuální početnosti.

In [ ]:
# PŘÍPRAVA DAT PRO GRAF 2
# Data s rozlišením státního občanství
df_cz = df_all[
    (df_all['vuzemi_cis'] == 97) & 
    (df_all['pohlavi_txt'].isna()) & 
    (df_all['vek_txt'].isna()) &
    (df_all['stobcan_txt'].notna())      # notna() = chceme konkrétní státy (ne celkový součet)
].copy()

In [ ]:
# GRAF 2: Pět nejpočetnějších skupin cizinců
# seřadíme od největšího a vezmeme prvních 5 názvů zemí
top_zeme = df_cz[df_cz['rok'] == max_rok].groupby('stobcan_txt')['hodnota'].sum().sort_values(ascending=False).head(5).index.tolist()
# vybereme jen řádky, které patří těmto 5 zemím
df_top = df_cz[df_cz['stobcan_txt'].isin(top_zeme)]
# sloupce = Názvy zemí
# hodnoty = Počty osob
df_top_pivot = df_top.pivot_table(index='rok', columns='stobcan_txt', values='hodnota', aggfunc='sum')

fig, ax = plt.subplots(figsize=(12, 6))
df_top_pivot.plot(kind='line', ax=ax, linewidth=2.5, marker='o')
ax.set_title(f'Pět nejpočetnějších skupin cizinců v ČR (Top 5 v roce {max_rok})')
ax.set_ylabel('Počet osob')
ax.set_xlabel('Rok')
ax.legend(title='Státní občanství')
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.grid(True)
plt.tight_layout()
plt.show()

### Jak se vyvíjí počty dalších významných menšin?

Pokud bychom všechny země zobrazili v jednom grafu, dominantní státy (hlavně Ukrajina) by "zploštily" křivky menších komunit, které by se staly nečitelnými. Proto nyní ty největší skupiny vyloučíme a zaměříme se na detailní vývoj dalších 15 významných národností. Zde můžeme pozorovat trendy například u asijských komunit nebo zemí západní Evropy.

**Postup filtrace:**

1.  Vyloučili jsme 5 největších zemí z předchozího kroku.
2.  Vybrali jsme dalších 15 nejpočetnějších národností podle celkových součtů.
3.  Aplikovali jsme různé styly čar a značek pro lepší rozlišení mnoha křivek.
4.  **Výsledek jsme vizualizovali:**
    * **Barevné linie:** Vývoj pro 15 států (mimo Top 5).
    * **Osa Y:** Detailnější měřítko než v předchozím grafu.

In [ ]:
# PŘÍPRAVA DAT PRO GRAF 3
TOP_N = 15
exclude_countries = ['Ukrajina', 'Rusko', 'Vietnam', 'Slovensko', 'Rumunsko'] 


# Filtrujeme: Celá ČR (97), nerozlišené pohlaví/věk, s rozlišením státu
df_trend = df_all[
    (df_all['vuzemi_cis'] == 97) &
    (df_all['pohlavi_cis'].isna()) & 
    (df_all['vek_cis'].isna()) &
    (df_all['stobcan_txt'].notna())
].copy()

# Očištění názvů
df_trend['stobcan_txt'] = df_trend['stobcan_txt'].str.strip()

# Vytvoření pivot tabulky (řádky=roky, sloupce=státy)
df_pivot = df_trend.pivot_table(index='rok', columns='stobcan_txt', values='hodnota', aggfunc='sum').fillna(0)

# Odstranění vyloučených zemí
cols_to_drop = [c for c in df_pivot.columns if c in exclude_countries]
df_pivot = df_pivot.drop(columns=cols_to_drop)

# Výběr TOP N zemí podle celkového součtu za celé období
top_countries = df_pivot.sum().sort_values(ascending=False).head(TOP_N).index
df_plot = df_pivot[top_countries]

In [ ]:
# GRAF 3: Vývoj počtu cizinců podle státního občanství 
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(16, 10))

# Generátory pro styly (barvy, značky, čáry)
colors = plt.cm.tab20(np.linspace(0, 1, TOP_N))
markers = itertools.cycle(['o', 's', '^', 'D'])
linestyles = itertools.cycle(['-', '--', '-.', ':']) 

for i, country in enumerate(df_plot.columns):
    ax.plot(
        df_plot.index, 
        df_plot[country], 
        label=country,
        color=colors[i],            
        marker=next(markers),       
        linestyle=next(linestyles), 
        linewidth=2.5,
        markersize=7,               
        alpha=0.9,
        clip_on=False,  
        zorder=10       
    )

# Úprava os a popisků
ax.set_xlim(df_plot.index.min(), df_plot.index.max())
ax.set_title(f'Vývoj počtu cizinců v ČR (Top {TOP_N} - mimo {", ".join(exclude_countries)})', fontsize=16, pad=20)
ax.set_xlabel('Rok', fontsize=12)
ax.set_ylabel('Počet osob', fontsize=12)

# Osa X - pouze celé roky
ax.set_xticks(df_plot.index)
if len(df_plot.index) > 10:
    ax.tick_params(axis='x', rotation=45)

# Formátování osy Y (tisíce s mezerou)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Legenda vně grafu
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', title='Státní občanství')

plt.tight_layout()
plt.show()


### Kde cizinci v ČR žijí (kraje)?

Migrace není po republice rozprostřena rovnoměrně. Tento graf srovnává absolutní počty cizinců v jednotlivých krajích. Cílem je ukázat, jak výrazná je koncentrace v hlavním městě a Středočeském kraji oproti ostatním regionům, což má zásadní dopad na lokální infrastrukturu a služby.

**Postup analýzy:**

1.  Vyfiltrovali jsme data pouze pro aktuální rok a úroveň krajů (NUTS 3).
2.  Agregovali jsme sumy cizinců za každý kraj zvlášť.
3.  Seřadili jsme kraje sestupně od nejvyššího počtu po nejnižší.
4.  **Výsledek jsme vizualizovali:**
    * **Sloupce:** Absolutní počet cizinců v daném kraji.
    * **Popisky:** Přesná hodnota nad každým sloupcem pro snazší čtení.

In [ ]:
# PŘÍPRAVA DAT PRO GRAF 4
# Data pro kraje (aktuální rok)
df_regions = df_all[
    (df_all['rok'] == max_rok) &
    (df_all['vuzemi_cis'] == 101) &      # 101 = Kód pro Kraje (NUTS 3)
    (df_all['pohlavi_txt'].isna()) &
    (df_all['vek_txt'].isna()) &
    (df_all['stobcan_txt'].isna())
].copy()
df_regions_agg = df_regions.groupby('kraj_txt')['hodnota'].sum().reset_index()

In [ ]:
# GRAF 4: Srovnání krajů

# seřazení od největšího po nejmenší
df_regions_sorted = df_regions_agg.sort_values('hodnota', ascending=False)
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_title(f'Rozložení cizinců v krajích (Rok {max_rok})')
ax.set_ylabel('Počet osob')
ax.set_xlabel('Kraj')

ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
bars = ax.bar(df_regions_sorted['kraj_txt'], df_regions_sorted['hodnota'], color='#4c72b0')
# Přidání popisků nad sloupce pro kontrolu
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{int(height):,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## Interaktivní prostorová analýza
Následující mapy využívají knihovnu folium pro vizualizaci globálních a lokálních dat.

### Odkud k nám cizinci přicházejí? (Mapa světa)

Tabulky nám neřeknou vše. Pro lepší představu o globálním kontextu si vykreslíme interaktivní mapu světa. Barva země odpovídá počtu jejích občanů žijících v ČR.
*Poznámka: Pro lepší čitelnost je barevná škála oříznuta (nastaven limit), aby extrémně vysoké hodnoty u jedné země (např. Ukrajiny) nezpůsobily, že zbytek mapy bude vypadat prázdný.*

**Postup tvorby mapy:**

1.  Načetli jsme geometrii světových hranic (shapefile).
2.  Propojili jsme statistická data s mapou pomocí anglických názvů zemí (s opravami překlepů).
3.  Ořízli jsme maximální hodnoty (clip), aby jedna dominantní země "nezhasla" zbytek mapy.
4.  **Výsledek jsme vizualizovali:**
    * **Barevná škála (YlOrRd):** Čím tmavší červená, tím více osob z dané země žije v ČR.
    * **Interaktivita:** Po najetí myší se zobrazí přesný počet a název země.

In [ ]:
# PŘÍPRAVA DAT PRO MAPU SVĚTA
# Cesta k shapefilu světa
gdf_world = gpd.read_file('world-administrative-boundaries/world-administrative-boundaries.shp')

# Načtení statistických dat a číselníku
df = pd.read_csv('data/CIZ01_2024.CSV')
df_cis = pd.read_csv('CIS3228_EN.csv')

# Filtrace: Rok 2024, Celá ČR (97), bez rozlišení pohlaví/věku
df_2024 = df[
    (df['rok'] == 2024) & 
    (df['vuzemi_cis'] == 97) & 
    (df['pohlavi_cis'].isna()) & 
    (df['vek_cis'].isna()) & 
    (df['stobcan_kod'].notna())
].copy()

# Slovník: Kód země -> Český název
czech_names_map = df[['stobcan_kod', 'stobcan_txt']].dropna().drop_duplicates().set_index('stobcan_kod')['stobcan_txt']

# Propojení s anglickými názvy z číselníku
cis_countries = df_cis[df_cis['kodcis'] == 3228][['chodnota', 'zkrtext']].copy()
cis_countries['english_name'] = cis_countries['zkrtext']
cis_countries['chodnota'] = cis_countries['chodnota'].astype(float)

# Připojení anglických názvů přes číselný kód
df_merged = df_2024.merge(cis_countries, left_on='stobcan_kod', right_on='chodnota', how='left')
df_merged['czech_name'] = df_merged['stobcan_kod'].map(czech_names_map)

# OPRAVY NÁZVŮ (Sjednocení s mapou světa)
rename_map_wab = {
    'russian': 'russian federation', 
    'moldova': 'moldova, republic of',
    'united kingdom': 'u.k. of great britain and northern ireland',
    'bosnia and herzegovina': 'bosnia & herzegovina',
    'macedonia': 'the former yugoslav republic of macedonia',
    'libya': 'libyan arab jamahiriya',
    'sudan (the)': 'sudan',
    'tanzania': 'united republic of tanzania',
    'syria': 'syrian arab republic',
    'iran': 'iran (islamic republic of)',
    'south korea': 'republic of korea',
    'north korea': "democratic people's republic of korea",
    'laos': "lao people's democratic republic",
}

# Aplikace oprav
df_merged['english_name_clean'] = df_merged['english_name'].str.lower().str.strip().replace(rename_map_wab)
gdf_world['english_nam_clean'] = gdf_world['English_Nam'].str.lower().str.strip()

# Agregace dat za země
stats_world = df_merged.groupby(['english_name_clean', 'czech_name'])['hodnota'].sum().reset_index()

# Spojení s mapou
final_map_data = gdf_world.merge(stats_world, left_on='english_nam_clean', right_on='english_name_clean', how='left')

# Oříznutí pro barvení (Ukrajina a spol. budou mít max 50 000)
final_map_data['hodnota_pro_barvu'] = final_map_data['hodnota'].clip(upper=50000)

# Státům bez dat se přiřadí 0 místo NaN
final_map_data['hodnota_pro_barvu'] = final_map_data['hodnota_pro_barvu'].fillna(0)
final_map_data['hodnota'] = final_map_data['hodnota'].fillna(0)

# Doplnění popisků pro tooltip (pokud chybí český název, použije se anglický)
final_map_data['display_name'] = final_map_data['czech_name'].fillna(final_map_data['English_Nam'])

In [ ]:
# VYKRESLENÍ MAPY 1: Cizinci v ČR podle občanství

m1 = folium.Map(location=[49, 15], zoom_start=4, tiles='CartoDB positron')

# Barevná vrstva
folium.Choropleth(
    geo_data=final_map_data,
    data=final_map_data,
    columns=['English_Nam', 'hodnota_pro_barvu'],
    key_on='feature.properties.English_Nam',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Počet cizinců v ČR podle státního občanství 2024 (červená = 50 000 a více)', 
    nan_fill_color='#f0f0f0',
    bins=5, 
    highlight=True
).add_to(m1)

# Interaktivní vrstva
folium.features.GeoJson(
    final_map_data,
    style_function=lambda x: {'color': 'transparent', 'fillColor': 'transparent', 'weight': 0},
    tooltip=folium.features.GeoJsonTooltip(
        fields=['display_name', 'hodnota'], 
        aliases=['Země:', 'Počet osob:'],
        style="background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;",
        localize=True
    )
).add_to(m1)

m1


### Jaký je poměr mužů a žen podle zemí?
Poměr pohlaví nám často prozradí motivaci k migraci. Převaha mužů obvykle indikuje pracovní migraci (těžký průmysl, stavebnictví), zatímco vyrovnaný poměr nebo převaha žen může naznačovat rodinnou migraci, práci ve službách nebo útěk před válkou.

**Postup výpočtu:**

1.  Vypočítali jsme procentuální podíl mužů pro každou zemi (`muži / celkem * 100`).
2.  Rozdělili jsme data do intervalů zvýrazňujících extrémy.
3.  **Výsledek jsme vizualizovali:**
    * **Modře:** Převaha mužů (nad 55 %).
    * **Bíle:** Vyrovnaný poměr (cca 50:50).
    * **Červeně:** Převaha žen (pod 45 %).

In [ ]:
# PŘÍPRAVA DAT PRO MAPU 2 (Poměr mužů a žen)

# Hledáme řádky, kde je vyplněno pohlaví a zároveň jsou to konkrétní státy
df_gender = df[
    (df['rok'] == 2024) & (df['vuzemi_cis'] == 97) & 
    (df['pohlavi_cis'].notna()) & (df['vek_cis'].isna()) & 
    (df['stobcan_kod'].notna())
].copy()

# Tabulka -> chceme sloupce 'muž', 'žena' vedle sebe
pivot_gender = df_gender.pivot_table(index='stobcan_kod', columns='pohlavi_txt', values='hodnota', aggfunc='sum').reset_index()

# Ošetření chybějících sloupců (kdyby náhodou chyběli muži nebo ženy úplně)
if 'muž' not in pivot_gender.columns: pivot_gender['muž'] = 0
if 'žena' not in pivot_gender.columns: pivot_gender['žena'] = 0
pivot_gender = pivot_gender.fillna(0)

# Výpočet procenta mužů
pivot_gender['celkem'] = pivot_gender['muž'] + pivot_gender['žena']
# Pokud je celkem > 0, vypočítáme podíl, jinak 0
pivot_gender['pct_muz'] = (np.where(pivot_gender['celkem'] > 0, (pivot_gender['muž'] / pivot_gender['celkem']) * 100, 0)).round(1)

# Připojení názvů a oprava názvů
gender_merged = pivot_gender.merge(cis_countries, left_on='stobcan_kod', right_on='chodnota', how='left')
gender_merged['czech_name'] = gender_merged['stobcan_kod'].map(czech_names_map)

# Aplikujeme stejný slovník oprav z předchozí části
gender_merged['english_name_clean'] = gender_merged['english_name'].str.lower().str.strip().replace(rename_map_wab)

# Spojení s mapou světa
map2_data = gdf_world.merge(gender_merged, left_on='english_nam_clean', right_on='english_name_clean', how='left')

# Popisek pro tooltip
map2_data['display_name'] = map2_data['czech_name'].fillna(map2_data['English_Nam'])

# Doplnění nul pro tooltip
map2_data['muž'] = map2_data['muž'].fillna(0)
map2_data['žena'] = map2_data['žena'].fillna(0)


In [ ]:
# VYKRESLENÍ MAPY 2: Poměr mužů a žen
m2 = folium.Map(location=[49, 15], zoom_start=4, tiles='CartoDB positron')

# Barevná vrstva
folium.Choropleth(
    geo_data=map2_data, 
    data=map2_data,
    columns=['English_Nam', 'pct_muz'],
    key_on='feature.properties.English_Nam',
    fill_color='RdBu',
    fill_opacity=0.7, 
    line_opacity=0.2,
    legend_name='Podíl mužů mezi cizinci (%) v roce 2024 - (červená < 50% ženy, modrá > 50% muži)',
    nan_fill_color='#f0f0f0',
    # Intervaly: Zvýrazníme extrémy, střed (45-55) je vyvážený
    bins=[0, 30, 45, 55, 70, 100], 
    highlight=True
).add_to(m2)

# Interaktivní vrstva
folium.features.GeoJson(
    map2_data, 
    style_function=lambda x: {'color': 'transparent', 'fillColor': 'transparent', 'weight': 0},
    tooltip=folium.features.GeoJsonTooltip(
        fields=['display_name', 'pct_muz', 'muž', 'žena'],
        aliases=['Země:', 'Podíl mužů (%):', 'Počet mužů:', 'Počet žen:'],
        localize=True
    )
).add_to(m2)

m2


### Které okresy jsou nejvíce zatížené?
Absolutní čísla mohou být zavádějící – velký okres bude mít přirozeně více cizinců. Abychom viděli skutečný sociální dopad, přepočítáme data na relativní hustotu (počet cizinců na 1000 obyvatel). Tato mapa nejlépe ukazuje, které konkrétní okresy a průmyslové zóny jsou migrací nejvíce ovlivněny.

**Postup analýzy:**

1.  Načetli jsme hranice okresů ČR a data o počtu jejich obyvatel.
2.  Spojili jsme data o cizincích s jednotlivými okresy.
3.  Vypočítali jsme indikátor: `(Počet cizinců / Počet obyvatel) * 1000`.
4.  **Výsledek jsme vizualizovali:**
    * **Barva okresu:** Indikuje počet cizinců na 1000 obyvatel.
    * **Tooltip:** Zobrazuje název okresu a přesnou hustotu.

In [ ]:
# PŘÍPRAVA DAT PRO MAPU 3 (Hustota v okresech)

# Definice cesty k mapě
path_okresy = 'shp/Okresy.shp' 

# 1. NAČTENÍ A OPRAVA SHAPEFILU
gdf_okresy = gpd.read_file(path_okresy)

# Oprava formátu datumu (Folium má problém s datovými typy datetime)
for col in gdf_okresy.columns:
    if pd.api.types.is_datetime64_any_dtype(gdf_okresy[col]) or pd.api.types.is_object_dtype(gdf_okresy[col]):
        try:
            gdf_okresy[col] = gdf_okresy[col].astype(str)
        except:
            pass

# 2. FILTRACE DAT
# Rok 2024, Okresy (101), Celkem (bez rozlišení)
df_okresy_raw = df_all[
    (df_all['rok'] == 2024) & 
    (df_all['vuzemi_cis'] == 101) & 
    (df_all['stobcan_cis'].isna()) & 
    (df_all['pohlavi_cis'].isna()) & 
    (df_all['vek_cis'].isna())
].copy()

# 3. OPRAVA NÁZVU PRAHY
# V datech je "Praha", v mapě "Hlavní město Praha"
df_okresy_raw['vuzemi_txt_clean'] = df_okresy_raw['vuzemi_txt'].replace({'Praha': 'Hlavní město Praha'})

# 4. AGREGACE
stats_okresy = df_okresy_raw.groupby('vuzemi_txt_clean')['hodnota'].sum().reset_index()

# 5. PROPOJENÍ S MAPOU
# Používáme sloupec 'nazev' z shapefilu a 'vuzemi_txt_clean' z dat
map3_data = gdf_okresy.merge(
    stats_okresy, 
    left_on='nazev', 
    right_on='vuzemi_txt_clean', 
    how='left'
)

# Kde chybí data dáme 0
map3_data['hodnota'] = map3_data['hodnota'].fillna(0)

# 6. VÝPOČET HUSTOTY
# Výpočet: (Počet cizinců / Počet obyvatel) * 1000
map3_data['hustota'] = map3_data.apply(
    lambda row: round((row['hodnota'] / row['Poc_obyv_S'] * 1000), 1) if row['Poc_obyv_S'] > 0 else 0, 
    axis=1
)

col_to_plot = 'hustota'
legend_title = 'Počet cizinců na 1000 obyvatel'


In [ ]:
# VYKRESLENÍ MAPY 3

m3 = folium.Map(location=[49.8, 15.5], zoom_start=7, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=map3_data, 
    data=map3_data,
    columns=['nazev', 'hustota'],    
    key_on='feature.properties.nazev', 
    fill_color='YlOrRd', 
    fill_opacity=0.7, 
    line_opacity=0.2,
    legend_name='Počet cizinců na 1000 obyvatel (2024)',
    highlight=True, 
    bins=7
).add_to(m3)

folium.features.GeoJson(
    map3_data, 
    style_function=lambda x: {'color': 'transparent', 'fillColor': 'transparent'},
    tooltip=folium.features.GeoJsonTooltip(
        fields=['nazev', 'hodnota', 'hustota'], 
        aliases=['Okres:', 'Celkem cizinců:', 'Na 1000 obyv.:'], 
        localize=True
    )
).add_to(m3)

display(m3)

### Jak se měnilo osídlení Ukrajinců v čase?

Statická mapa nám neukáže, *kdy* ke změnám došlo. Proto si nyní vytvoříme interaktivní přehrávač, který nám umožní sledovat vývoj v čase rok po roce. Zaměříme se na konkrétní vybranou zemi (např. Ukrajinu), abychom viděli, jak se měnilo usídlování jejích občanů v českých okresech v průběhu posledních dvou dekád.

**Postup tvorby animace:**

1.  Vyfiltrovali jsme data pouze pro občany Ukrajiny na úrovni okresů.
2.  V cyklu jsme vygenerovali mapu pro každý rok zvlášť a uložili ji do paměti.
3.  Vytvořili jsme interaktivní přehrávač (widget).
4.  **Ovládání:**
    * **Slider/Play:** Umožňuje posun v čase (2004–2024).
    * **Mapa:** Ukazuje zahušťování populace v konkrétních regionech.

In [ ]:
# --- NASTAVENÍ ---
TARGET_COUNTRY = 'UKRAJINA'
# Načtení mapy okresů 
SHP_PATH = 'shp/Okresy.shp' 

# Intervaly pro legendu
BINS = [0, 50, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 150000]

okresy_shp = gpd.read_file(SHP_PATH, encoding='utf-8')

cmap = plt.get_cmap('YlOrRd', len(BINS)-1)
norm = mcolors.BoundaryNorm(BINS, cmap.N)

image_cache = {}
# Získáme roky přímo z dat
years = sorted(df_all['rok'].unique())

plt.rcParams['font.family'] = 'sans-serif'

# Vyfiltrujeme si data pro danou zemi najednou z df_all
df_country = df_all[
    (df_all['stobcan_txt'].str.upper() == TARGET_COUNTRY) & 
    (df_all['vuzemi_cis'] == 101) &       # Pouze okresy
    (df_all['pohlavi_txt'].isna()) &      # Bez rozlišení pohlaví
    (df_all['vek_txt'].isna())            # Bez rozlišení věku
].copy()

# Sjednocení názvu Prahy
df_country['vuzemi_txt'] = df_country['vuzemi_txt'].replace({'Praha': 'Hlavní město Praha'})

for rok in years:
    
    # Filtrace pro konkrétní rok
    stats = df_country[df_country['rok'] == rok].groupby('vuzemi_txt')['hodnota'].sum().reset_index()
    
    if stats.empty: continue

    # Merge s mapou
    merged = okresy_shp.merge(stats, left_on='nazev', right_on='vuzemi_txt', how='left').fillna(0)
    
    # Vykreslení
    fig, ax = plt.subplots(1, 1, figsize=(12, 9))
    okresy_shp.plot(ax=ax, color='#f0f0f0', edgecolor='#aaaaaa', linewidth=0.4)
    
    data_plot = merged[merged['hodnota'] > 0]
    
    data_plot.plot(
        ax=ax, column='hodnota', cmap=cmap, norm=norm, 
        legend=False, edgecolor='#888888', linewidth=0.4
    )
    
    # Legenda
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm._A = []
    cbar = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, extend='max')
    cbar.set_label(f'Počet občanů ({TARGET_COUNTRY})', size=12)
    cbar.set_ticks(BINS)
    cbar.set_ticklabels([f'{int(b)}' for b in BINS])

    ax.set_title(f'Vývoj počtu občanů - {TARGET_COUNTRY}\nRok: {rok}', fontsize=18, fontweight='bold', pad=20)
    ax.set_axis_off()
    
    # Uložení do RAM
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=80, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    image_cache[rok] = buf.read()



# --- WIDGETY ---
first_year = min(image_cache.keys())
image_widget = widgets.Image(value=image_cache[first_year], format='png', width=850, layout=widgets.Layout(margin='10px 0px'))
slider = widgets.IntSlider(min=min(years), max=max(years), step=1, value=min(years), description='Rok:', layout=widgets.Layout(width='500px'))
play = widgets.Play(value=min(years), min=min(years), max=max(years), interval=1000, description="Přehrát", repeat=False)
widgets.jslink((play, 'value'), (slider, 'value'))

def on_year_change(change):
    rok = change['new']
    if rok in image_cache:
        image_widget.value = image_cache[rok]
slider.observe(on_year_change, names='value')

controls = widgets.HBox([play, slider], layout=widgets.Layout(justify_content='center', margin='10px'))
display(widgets.VBox([image_widget, controls], layout=widgets.Layout(align_items='center', border='1px solid #ddd', padding='20px')))


### Interaktivní průzkum globální demografie
Na závěr si spojíme geografická data s časovou osou. Následující nástroj vám umožní procházet mapy světa v různých letech a sledovat, jak se měnila struktura migrantů (poměr mužů a žen) globálně. Můžete tak sami objevit momenty, kdy u konkrétních zemí došlo ke změně charakteru migrace.

**Postup:**

1.  Předpočítali jsme interaktivní mapy pro všechny dostupné roky.
2.  Vložili jsme je do výstupního widgetu s posuvníkem.
3.  **Výsledek:**
    * Uživatel může měnit rok a sledovat, jak se mění barva zemí (struktura migrace) v čase.

In [ ]:

# 1. PŘÍPRAVA DAT 

# Filtrace dat
df_gender_ts = df_all[
    (df_all['rok'] >= 2004) & 
    (df_all['rok'] <= 2024) &
    (df_all['vuzemi_cis'] == 97) & 
    (df_all['pohlavi_cis'].notna()) & 
    (df_all['vek_cis'].isna()) & 
    (df_all['stobcan_kod'].notna())
].copy()

# Pivot tabulka
pivot_ts = df_gender_ts.pivot_table(
    index=['rok', 'stobcan_kod'], 
    columns='pohlavi_txt', 
    values='hodnota', 
    aggfunc='sum'
).reset_index()

# Úklid sloupců
pivot_ts.columns = [str(c).lower() for c in pivot_ts.columns]
if 'muž' not in pivot_ts.columns: pivot_ts['muž'] = 0
if 'žena' not in pivot_ts.columns: pivot_ts['žena'] = 0
pivot_ts = pivot_ts.fillna(0)

# Výpočet podílu
pivot_ts['celkem'] = pivot_ts['muž'] + pivot_ts['žena']
pivot_ts['pct_muz'] = np.where(pivot_ts['celkem'] > 0, (pivot_ts['muž'] / pivot_ts['celkem']) * 100, 0)

# Merge s číselníky
pivot_ts['stobcan_kod'] = pd.to_numeric(pivot_ts['stobcan_kod'], errors='coerce')
merged_ts = pivot_ts.merge(cis_countries, left_on='stobcan_kod', right_on='chodnota', how='left')
merged_ts['english_name_clean'] = merged_ts['english_name'].str.lower().str.strip().replace(rename_map_wab)
merged_ts['czech_name'] = merged_ts['stobcan_kod'].map(czech_names_map)


In [ ]:

# 2. FUNKCE PRO GENEROVÁNÍ MAPY 
def create_map_object(rok):
    # Filtrace
    data_rok = merged_ts[merged_ts['rok'] == rok].copy()
    
    # Merge s geometrií
    map_data = gdf_world.merge(data_rok, left_on='english_nam_clean', right_on='english_name_clean', how='left')
    
    # Tooltip data
    map_data['display_name'] = map_data['czech_name'].fillna(map_data['English_Nam'])
    map_data['pct_muz_val'] = map_data['pct_muz'].fillna(0).round(1)
    map_data['muž_val'] = map_data['muž'].fillna(0).astype(int)
    map_data['žena_val'] = map_data['žena'].fillna(0).astype(int)
    
    # Vytvoření mapy
    m = folium.Map(location=[25, 10], zoom_start=2, tiles='CartoDB positron')
    
    folium.Choropleth(
        geo_data=map_data,
        data=map_data,
        columns=['English_Nam', 'pct_muz'],
        key_on='feature.properties.English_Nam',
        fill_color='RdBu',
        fill_opacity=0.7,
        line_opacity=0.2,
        nan_fill_color='#f0f0f0',
        bins=[0, 30, 45, 55, 70, 100], 
        legend_name=f'Podíl mužů (%) v roce {rok}',
        highlight=True
    ).add_to(m)
    
    folium.features.GeoJson(
        map_data,
        style_function=lambda x: {'color': 'transparent', 'fillColor': 'transparent', 'weight': 0},
        tooltip=folium.features.GeoJsonTooltip(
            fields=['display_name', 'pct_muz_val', 'muž_val', 'žena_val'],
            aliases=['Země:', 'Podíl mužů (%):', 'Muži:', 'Ženy:'],
            localize=True
        )
    ).add_to(m)
    
    return m

In [ ]:
# 3. PŘEDGENERUJEME VŠECHNY MAPY DO PAMĚTI 
years = sorted(merged_ts['rok'].unique())
available_years = [y for y in years if 2004 <= y <= 2024]

map_cache = {} 
 
for i, rok in enumerate(available_years):
    map_cache[rok] = create_map_object(rok)

In [ ]:
# 4. NASTAVENÍ SLIDERU A VÝSTUPU
out = widgets.Output()

slider = widgets.IntSlider(
    min=min(available_years), 
    max=max(available_years), 
    step=1, 
    value=max(available_years), 
    description='Rok:',
    layout=widgets.Layout(width='600px')
)

def on_year_change(change):
    rok = change['new']
    with out:
        clear_output(wait=True)

        display(map_cache[rok])


slider.observe(on_year_change, names='value')

# Zobrazit prvotní stav
with out:
    display(map_cache[max(available_years)])

display(widgets.VBox([slider, out]))

### Hraje roli vzdálenost od hranic? (Sousední státy)

U občanů sousedních států (Německo, Polsko, Rakousko, Slovensko) testujeme hypotézu, zda preferují bydlení v blízkosti své domoviny (např. kvůli dojíždění za prací či rodinou), nebo zda se stěhují hlouběji do vnitrozemí.

**Postup:**

1.  **Geometrický výpočet:** Pro každý okres v ČR jsme určili jeho střed (centroid) a vypočítali přesnou vzdálenost k hranicím daného souseda.
2.  **Spojení dat:** K prostorovým datům jsme připojili statistiky o počtu cizinců z roku 2024 a přepočítali je na hustotu (počet na 1000 obyvatel).
3.  **Korelace:** Pomocí lineární regrese (metoda nejmenších čtverců) hledáme trend mezi vzdáleností a hustotou osídlení.
4.  **Výsledek jsme vizualizovali:**
    * **Osa X:** Vzdálenost okresu od státní hranice (km).
    * **Osa Y:** Hustota cizinců dané národnosti.
    * **Červený bod:** Praha.
    * **Černá čára:** Trend. Pokud klesá, potvrzuje to, že s rostoucí vzdáleností od domova počet cizinců ubývá.

In [ ]:
# --- 1. PŘÍPRAVA DAT ---
sledovane_staty = {
    'Německo': 'Germany',
    'Polsko': 'Poland',
    'Rakousko': 'Austria',
    'Slovensko': 'Slovakia'
}

# Geometrie
gdf_sousede = gdf_world[gdf_world['English_Nam'].isin(sledovane_staty.values())].to_crs(epsg=5514)
gdf_okresy = gpd.read_file('shp/Okresy.shp').to_crs(epsg=5514)
gdf_okresy['centroid'] = gdf_okresy.geometry.centroid

def get_distance_km(point, geom):
    return point.distance(geom) / 1000

# Filtrace dat cizinců (2024)
df_2024 = df_all[
    (df_all['rok'] == 2024) &
    (df_all['vuzemi_cis'] == 101) &
    (df_all['pohlavi_txt'].isna()) & 
    (df_all['vek_txt'].isna())
].copy()
df_2024['vuzemi_txt_clean'] = df_2024['vuzemi_txt'].replace({'Praha': 'Hlavní město Praha'})


# --- 2. GENEROVÁNÍ GRAFŮ S VLASTNÍM TOOLTIPEM ---

for cesky_nazev, anglicky_nazev in sledovane_staty.items():
    
    # A) Výpočet vzdálenosti 
    geom_stat = gdf_sousede[gdf_sousede['English_Nam'] == anglicky_nazev].geometry.union_all()
       
    col_dist = 'vzdalenost_km'
    
    gdf_plot = gdf_okresy.copy()
    gdf_plot[col_dist] = gdf_plot['centroid'].apply(lambda x: get_distance_km(x, geom_stat)).round(1)
    
    # B) Spojení s daty
    df_stat = df_2024[df_2024['stobcan_txt'] == cesky_nazev]
    merged = gdf_plot.merge(df_stat, left_on='nazev', right_on='vuzemi_txt_clean')
    
    # C) Příprava dat
    merged['Legenda'] = merged['nazev'].apply(lambda x: 'Praha' if x == 'Hlavní město Praha' else 'Ostatní okresy')
    merged['pocet_osob'] = merged['hodnota'].astype(int)
    merged['hustota'] = ((merged['hodnota'] / merged['Poc_obyv_S']) * 1000).round(2)
    
    # D) Vykreslení
    fig = px.scatter(
        merged, 
        x=col_dist, 
        y='hustota',
        color='Legenda',
        
        # Data, která chceme mít dostupná v bublině 
        hover_data=['pocet_osob'],
        hover_name='nazev',
        
        # Barvy a řazení
        color_discrete_map={'Praha': 'red', 'Ostatní okresy': '#4c72b0'}, 
        category_orders={'Legenda': ['Praha', 'Ostatní okresy']},
        
        # Trendová čára
        trendline="ols",
        trendline_scope="overall",
        trendline_color_override="black",
        
        title=f'Vliv vzdálenosti od hranic: {cesky_nazev}',
        height=550
    )
    
    # E) VLASTNÍ ŠABLONA BUBLINY 
    # <extra></extra> na konci schová  nepotřebné informace (např. název stopy)
    fig.update_traces(
        hovertemplate=(
            "<b>%{hovertext}</b><br>" +             # Tučný název okresu
            "Vzdálenost = %{x:.1f} km<br>" +        # Vzdálenosti
            "Počet cizinců: %{customdata[0]:,}" +   # Počet osob
            "<extra></extra>"                       # Schová zbytek
        )
    )
    
    # F) Layout
    fig.update_layout(
        title_x=0.5,
        legend_title_text='',
        xaxis_title=f"Vzdálenost od hranic - {cesky_nazev} (km)",
        yaxis_title="Počet cizinců na 1000 obyvatel",
        plot_bgcolor='white',
        hoverlabel=dict(
            bgcolor="white",
            font_size=14,
            font_family="Arial"
        )
    )
    
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#f0f0f0')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#f0f0f0')

    # Přejmenování Trendové čáry
    for trace in fig.data:
        if trace.mode == 'lines':
            trace.name = 'Trend'
            trace.showlegend = True
    
    fig.show()